In [ ]:
import os
import shutil
import subprocess
from datasets import load_dataset

# Load the SWE Bench Lite test dataset
swebench = load_dataset('princeton-nlp/SWE-bench_Lite', split='test')

# Base folder to store all cloned repositories
base_folder = "codebases"
os.makedirs(base_folder, exist_ok=True)

def run_command(command, cwd=None):
    """Run a shell command in a given working directory."""
    result = subprocess.run(command, cwd=cwd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error running command: {command}\n{result.stderr}")
    return result

for task in swebench:
    instance_id = task['instance_id']
    repo_field = task['repo']
    base_commit = task['base_commit']

    # If the repo field doesn't look like a URL, convert it to one.
    # For example, "astropy/astropy" becomes "https://github.com/astropy/astropy.git"
    if not (repo_field.startswith("http://") or repo_field.startswith("https://")):
        repo_url = f"https://github.com/{repo_field}.git"
    else:
        repo_url = repo_field

    # Create a directory for this instance
    instance_folder = os.path.join(base_folder, instance_id)
    os.makedirs(instance_folder, exist_ok=True)

    # Clone the repository into the instance folder
    print(f"Cloning {repo_url} into {instance_folder}...")
    clone_cmd = f"git clone {repo_url} ."
    clone_result = run_command(clone_cmd, cwd=instance_folder)
    if clone_result.returncode != 0:
        print(f"Failed to clone repository {repo_url} for instance {instance_id}. Skipping...")
        continue

    # Checkout the repository to the specified commit
    print(f"Checking out commit {base_commit} in {instance_folder}...")
    checkout_cmd = f"git checkout {base_commit}"
    checkout_result = run_command(checkout_cmd, cwd=instance_folder)
    if checkout_result.returncode != 0:
        print(f"Failed to checkout commit {base_commit} in {instance_folder}.")
        continue

    # Delete the .git folder to save space
    git_dir = os.path.join(instance_folder, ".git")
    if os.path.exists(git_dir):
        print(f"Deleting .git folder in {instance_folder}...")
        shutil.rmtree(git_dir)
    print(f"Finished processing instance {instance_id}.\n")